# Lab 3.1 &mdash; Your First Graph

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Declare workflow state as a <code>TypedDict</code>
- Write a node &mdash; a plain function, state in, <i>partial</i> state out
- Wire <code>START</code> and <code>END</code>, then <code>compile()</code> and <code>invoke()</code>

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

`create_agent` from Module 1 is one fixed loop: model, tools, repeat. When the control flow is
yours &mdash; branch here, go round again there, stop for a person &mdash; you want the thing
underneath it.

A `StateGraph` has three parts and no more:

| Part | What it is |
|---|---|
| **state** | a `TypedDict`. One shared object every node reads and writes |
| **nodes** | plain functions: `state -> partial state` |
| **edges** | what runs next. `START` is the way in, `END` the way out |

**There is no model in this lab.** That is the point: the graph is control flow, and control flow
is ordinary software you can test. The model goes *inside one node*, from Lab 3.5 on.

## Section 1 &mdash; State, and a node that returns part of it

A node gets the **whole** state and returns **only the keys it changed**. LangGraph merges the rest.

Returning the whole state is the classic first mistake: it works until two nodes run and one
writes back a stale copy of what the other just changed. Both candidates are written out below.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class LeaveState(TypedDict):
    request_id: str
    summary: str
    notes: list


def summarise(state: LeaveState) -> dict:
    r = REQUESTS[state["request_id"]]
    line = f'{r["who"]}: {r["days"]} day(s) of {r["kind"]} leave'

    whole_state  = {**state, "summary": line, "notes": state["notes"] + ["summarise"]}
    only_changed = {"summary": line, "notes": state["notes"] + ["summarise"]}

    return only_changed   # a node returns ONLY the keys it changed

In [ ]:
# --- Self-check: Section 1
check("summarise() fills in a readable summary",
      lambda: "Priya Nair" in summarise({"request_id": "LV-5001", "notes": []})["summary"])
check("summarise() returns ONLY the keys it changed",
      lambda: set(summarise({"request_id": "LV-5001", "notes": []})) == {"summary", "notes"},
      "request_id was not changed by this node, so it should not be in the return value")
score()

## Section 2 &mdash; Edges, compile, invoke

Four lines build it and one runs it. `START` and `END` are ordinary edge endpoints, not settings.

In [ ]:
def build_graph():
    builder = StateGraph(LeaveState)
    builder.add_node("summarise", summarise)
    builder.add_edge(START, "summarise")   # START is the way in
    builder.add_edge("summarise", END)     # END is the way out
    return builder.compile()

In [ ]:
# --- Self-check: Section 2   (a real compiled graph, really invoked -- no model in it)
def run(rid):
    return build_graph().invoke({"request_id": rid, "summary": "", "notes": []})

check("the graph compiles and runs end to end",
      lambda: run("LV-5001")["summary"].startswith("Priya Nair"),
      "both edges are needed: START -> summarise -> END")
check("state you did not touch survives the run",
      lambda: run("LV-5004")["request_id"] == "LV-5004",
      "LangGraph merged request_id through for you -- that is what partial state buys")
score()

## Watch it run

In [ ]:
app = guard(build_graph)

if app is not None:
    for rid in ["LV-5001", "LV-5004"]:
        out = app.invoke({"request_id": rid, "summary": "", "notes": []})
        print(f"{rid} -> {out['summary']}   notes={out['notes']}")
    g = app.get_graph()
    print("\nthe graph you just built:")
    for e in g.edges:
        print(f"   {e.source} -> {e.target}")

### Read it

The state came back whole &mdash; your node returned two keys and `request_id` is still there, and
you wrote no merge code. And the graph is deterministic: same input, same output, every time.
Put it under a unit test and it will never flake.

In [ ]:
score()

## Your turn

1. Add a second node, `stamp_policy`, that appends `" -- needs manager"` to `summary` when `days`
   exceeds `POLICY["manager_over_days"]`. Wire `START -> summarise -> stamp_policy -> END`.
2. Delete the `add_edge(START, ...)` line and re-compile. Read the error once, deliberately,
   rather than at 4pm on Day 3.